In [1]:
!pip install -q -U google-generativeai pillow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 46.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.


In [2]:
import google.generativeai as genai
import json
import os
import pathlib
import time
from tqdm.auto import tqdm
from PIL import Image, ImageFile

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [3]:
# 0. Mount Google Drive (ADD THIS HERE)
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [4]:
# 1. Setup API Key from Colab Secrets
from google.colab import userdata
gemini_token = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=gemini_token)

# Using Gemini 2.5 Flash (Free Tier)
model = genai.GenerativeModel("gemini-2.5-flash")

In [5]:
# Force PIL to load partially corrupted/truncated downloaded images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# 2. Minimalist Preprocessing
import io

def preprocess_for_vlm(image):
    # 1. Convert everything to standard RGB to drop problematic alpha channels
    if image.mode != 'RGB':
        image = image.convert('RGB')

    # 2. Save to an in-memory byte buffer as a standard JPEG
    byte_stream = io.BytesIO()
    # quality=95 ensures the menu text remains sharp and readable for the model
    image.save(byte_stream, format='JPEG', quality=95)

    # 3. Pass the raw bytes directly to Gemini, completely bypassing SDK bugs
    return {
        "mime_type": "image/jpeg",
        "data": byte_stream.getvalue()
    }

In [ ]:
# 3. Define the exact JSON structure
menu_schema = {
    "type": "OBJECT",
    "properties": {
        "items": {
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "name": {"type": "STRING"},
                    "price": {"type": "STRING"}
                },
                "required": ["name", "price"]
            }
        }
    },
    "required": ["items"]
}


# 4. The Prompt
prompt = """Extract every food or drink item and its price from this menu image.

Rules:
- Keep item names and prices exactly as written in the image.
- If an item has multiple sizes, list each size as a separate entry like "بيتزا مارجريتا (صغير)" and "بيتزا مارجريتا (كبير)".
- Only include items that have a visible price. Skip anything without a price.

Return a JSON object with one key "items" containing an array. Each item has "name" (string) and "price" (string)."""

In [7]:
# 5. Define input and output folders
image_folder = pathlib.Path("/gdrive/MyDrive/data")
output_folder = pathlib.Path("/gdrive/MyDrive/menu_json_ground_truth")
output_folder.mkdir(parents=True, exist_ok=True)

# Gather all images regardless of format
valid_extensions = {'.jpg', '.jpeg', '.png', '.webp'}
image_paths = [
    path for path in image_folder.iterdir()
    if path.suffix.lower() in valid_extensions
]

print(f"Found {len(image_paths)} images to process.")

Found 334 images to process.


In [ ]:
# 6. Processing Loop
for img_path in tqdm(image_paths, desc="Processing Menus"):
    # Check if JSON already exists to skip and save time/API credits
    json_path = output_folder / f"{img_path.stem}.json"
    if json_path.exists():
        continue

    try:
        raw_img = Image.open(img_path)
        processed_img = preprocess_for_vlm(raw_img)

        response = model.generate_content(
            [prompt, processed_img],
            generation_config=genai.GenerationConfig(
                response_mime_type="application/json",
                response_schema=menu_schema,
                temperature=0.0,
            )
        )

        with open(json_path, "w", encoding="utf-8") as f:
            parsed_json = json.loads(response.text)
            json.dump(parsed_json, f, ensure_ascii=False, indent=2)

        # Respect the 15 RPM limit
        time.sleep(4.1)

    except Exception as e:
        print(f"\nError processing {img_path.name}: {e}")
        if "429" in str(e):
            print("Hit rate limit. Pausing for 60 seconds...")
            time.sleep(60)

print("Extraction complete!")